# HetGNN Training-Data EDA & Feature-Quality Report

Interactive companion to `eda_feature_engineering.py`. Same logic, inline plots.

**Schema source of truth:** `gnn/src/utils/gnn_utils.py` — `NODE_TYPES`, `*_FEATURES`, `FEATURE_DIMS`, `ANOMALY_THRESHOLDS`, `EDGE_TYPES`.

**Snapshot fetch:** `gnn/src/utils/data.py` — `SpannerDataset` + `compute_temporal_features`.

**Cache:** loads from `notebook_artifacts/snapshots/` first (the same cache written by `test_model_training.py` / `test_training_and_serving_alvaro.ipynb`), so the EDA reflects exactly the data the model trained on. If absent, fetches a fresh window from Spanner.

## Sections
1. Setup
2. Load snapshots (cache-first)
3. Snapshot inventory
4. Build feature DataFrames
5. Feature quality report
6. Feature distributions
7. Temporal evolution
8. Per-node value heatmaps
9. Per-node zero-rate heatmaps
10. Down-event visualisations
11. Topology inventory + degree distribution
12. Per-router topology breakdown
13. Topology visualisation (networkx)
14. Node-presence completeness

## 1. Setup

In [ ]:
import datetime
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

# ── Paths: make gnn/src and gnn/tests importable ─────────────────────────────
_TESTS_DIR = Path.cwd().resolve()
_SRC_DIR   = _TESTS_DIR.parent / 'src'
for p in [str(_SRC_DIR), str(_TESTS_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Credentials ──────────────────────────────────────────────────────────────
_creds = str(_SRC_DIR / 'networkagent.json')
if os.path.exists(_creds):
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = _creds
os.environ.setdefault('GOOGLE_CLOUD_PROJECT', 'gnn-networks')

from utils.gnn_utils import ANOMALY_THRESHOLDS, FEATURE_DIMS, NODE_TYPES
import eda_feature_engineering as eda

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
print(f'NODE_TYPES:   {NODE_TYPES}')
print(f'FEATURE_DIMS: {FEATURE_DIMS}')
print(f'ANOMALY_THRESHOLDS (per-node reconstruction MSE): {ANOMALY_THRESHOLDS}')

## 2. Load snapshots (cache-first)

In [ ]:
PROJECT_ID       = os.environ['GOOGLE_CLOUD_PROJECT']
SPANNER_INSTANCE = os.getenv('SPANNER_INSTANCE', 'networktopology-instance')
SPANNER_DATABASE = os.getenv('SPANNER_DATABASE', 'networktopology-db')
INTERVAL_MINUTES = 0.5   # matches training notebook (30 s cadence)

# Shared training cache — populated by test_model_training.py / test_training_and_serving_alvaro.ipynb
SNAPSHOTS_DIR = Path('notebook_artifacts/snapshots')

# Fallback window if the cache is absent
FROM_TIME = datetime.datetime(2026, 5, 29, 20, 0, 0)
TO_TIME   = datetime.datetime(2026, 5, 29, 21, 0, 0)

snapshots = eda.load_snapshots(
    snapshots_dir   = SNAPSHOTS_DIR,
    from_time       = FROM_TIME,
    to_time         = TO_TIME,
    interval_minutes= INTERVAL_MINUTES,
    project         = PROJECT_ID,
    spanner_instance= SPANNER_INSTANCE,
    spanner_database= SPANNER_DATABASE,
)
print(f'\nLoaded {len(snapshots)} snapshot(s).')
if snapshots:
    print(f'Range: {snapshots[0]["timestamp"]}  \u2192  {snapshots[-1]["timestamp"]}')

## 3. Snapshot inventory

Per-snapshot counts: routers / interfaces / vpns / flows (with `up` ratio) and edge counts per relation.

In [ ]:
eda.print_snapshot_inventory(snapshots)

## 4. Build feature DataFrames

One long-form DataFrame per node type, one row per `(node, snapshot)`. Columns are the canonical features from `gnn_utils`.

In [ ]:
dfs = eda.extract_dataframes(snapshots)
for nt, df in dfs.items():
    n_nodes = df['id'].nunique() if not df.empty else 0
    print(f'  {nt:<10}  rows={len(df):>7,}  unique_nodes={n_nodes}')

## 5. Feature quality report

Per-feature: null %, zero %, mean, std, skew, kurtosis, constant / low-variance / outlier flags. Writes `eda_output/feature_quality.csv` and prints a per-node-type summary.

In [ ]:
OUT_DIR = Path('eda_output')
OUT_DIR.mkdir(exist_ok=True)
eda.report_feature_quality(dfs, OUT_DIR)
display(pd.read_csv(OUT_DIR / 'feature_quality.csv'))

## 6. Feature distributions

Continuous features \u2192 histograms with mean / median lines. Binary features \u2192 on-share bars.

In [ ]:
eda.plot_distributions(dfs, OUT_DIR)
for nt in NODE_TYPES:
    for fname in (f'distributions_{nt}.png', f'binary_{nt}.png'):
        p = OUT_DIR / fname
        if p.exists(): display(Image(str(p)))

## 7. Temporal evolution

Mean \u00b1 std band plus max line (max catches single-node spikes that the mean alone hides).

In [ ]:
eda.plot_temporal_evolution(dfs, OUT_DIR)
for nt in NODE_TYPES:
    p = OUT_DIR / f'temporal_{nt}.png'
    if p.exists(): display(Image(str(p)))

## 8. Per-node value heatmaps

Mean feature value per node \u00d7 feature. `router` / `vpn` show every node; `interface` / `flow` show the top 25 by total magnitude.

In [ ]:
eda.plot_per_node_heatmaps(dfs, OUT_DIR)
for nt in NODE_TYPES:
    p = OUT_DIR / f'heatmap_{nt}.png'
    if p.exists(): display(Image(str(p)))

## 9. Per-node zero-rate heatmaps

% of snapshots in which each feature was exactly 0, per node. A column of zeros for a single node usually means that target's Prometheus scrape is broken.

In [ ]:
eda.plot_zero_rate_heatmaps(dfs, OUT_DIR)
for nt in NODE_TYPES:
    p = OUT_DIR / f'zerorate_{nt}.png'
    if p.exists(): display(Image(str(p)))

## 10. Down-event visualisations

Router scatter (hostname \u00d7 time, marker on `state==0`) and an interface state-timeline heatmap (top 25 most-down interfaces).

In [ ]:
eda.plot_down_events(dfs, OUT_DIR)
for fname in ('down_events_router.png', 'down_events_interface.png'):
    p = OUT_DIR / fname
    if p.exists(): display(Image(str(p)))

## 11. Topology inventory + degree distribution

Edge counts per `EDGE_TYPES` relation in the latest snapshot, plus a node-degree boxplot per node type.

In [ ]:
eda.plot_topology(snapshots, OUT_DIR)
p = OUT_DIR / 'topology_degree.png'
if p.exists(): display(Image(str(p)))

## 12. Per-router topology breakdown

In [ ]:
eda.print_per_router_topology(snapshots)

## 13. Topology visualisation (networkx)

Router `peer_of` graph (size \u221d degree, colour = role, faded = `state=0`) and router-interface bipartite layout.

In [ ]:
eda.plot_topology_graph(snapshots, OUT_DIR)
for fname in ('topology_router_peer.png', 'topology_bipartite.png'):
    p = OUT_DIR / fname
    if p.exists(): display(Image(str(p)))

## 14. Node-presence completeness

Nodes appearing in fewer than 100 % of snapshots \u2014 flags ephemeral collectors, paused flows, dropped routers.

In [ ]:
eda.report_node_presence(dfs, snapshots)